# ПРЕАМБУЛА

Запустить полноценный `docker-compose` с Dify, Open WebUI и базами данных прямо внутри Google Colab физически невозможно (из-за ограничений среды на запуск Docker-демонов и нехватку RAM).

Однако для практического занятия мы создадим **архитектурный симулятор (Micro-Architecture Mock)**. Мы напишем Python-код, который в точности имитирует HTTP-взаимодействие между этими компонентами, а логику LangGraph обернем в локальный API.

### Шаг 0: Подготовка инфраструктуры (Инференс и Трейсинг)
Устанавливаем LangGraph, интеграции для локального LLM и **Arize Phoenix**.

In [1]:
# 1. Системные зависимости
!sudo apt-get update && sudo apt-get install -y zstd
# 2. Установка библиотек
!pip install -q langgraph langchain-community langchain-core arize-phoenix[experimental] openinference-instrumentation-langchain nest_asyncio open-webui
# 3. Установка Ollama
!curl -fsSL https://ollama.com/install.sh | sh

import os
import time
import subprocess

# Запуск Ollama в фоне
print("Запускаем Ollama...")
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)
print("Скачиваем модель qwen2.5-coder:1.5b...")
!ollama pull qwen2.5-coder:1.5b
print("✅ Ollama готова!")

# Запуск Open WebUI в фоне
print("Запускаем Open WebUI на порту 8080 (это займет около 10 секунд)...")
env = os.environ.copy()
env["OLLAMA_BASE_URL"] = "http://127.0.0.1:11434"
subprocess.Popen(["open-webui", "serve", "--port", "8080", "--host", "0.0.0.0"],
    env=env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
time.sleep(10)
print("✅ Open WebUI готов!")

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [85.2 kB]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 https://cli.github.com/packages stable/main amd64 Packages [355 B]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:10 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,786 kB]
Get:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease [24.6 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,836 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:14 h

### Шаг 1: Запуск Arize Phoenix (Observability)
В продакшене мы не можем использовать SaaS (LangSmith) из-за NDA. Мы поднимаем **Arize Phoenix** локально. Он перехватывает все вызовы к LLM и строит дерево трейсов (Spans).

In [2]:
import phoenix as px
import time
from phoenix.otel import register
from openinference.instrumentation.langchain import LangChainInstrumentor
import nest_asyncio

# 1. СНАЧАЛА ЗАПУСКАЕМ PHOENIX (До патча asyncio!)
print("Запуск Arize Phoenix...")
session = px.launch_app()
time.sleep(3) # Даем серверу uvicorn время запуститься в фоне

# 2. ТОЛЬКО ТЕПЕРЬ ПАТЧИМ ASYNCIO
print("Применяем патч nest_asyncio для работы LangGraph...")
nest_asyncio.apply()

# 3. Регистрируем OpenTelemetry провайдер
tracer_provider = register(
    project_name="langgraph-contract-reviewer",
    endpoint="http://127.0.0.1:6006/v1/traces"
)

# 4. Инструментируем (внедряем хуки)
LangChainInstrumentor().instrument(tracer_provider=tracer_provider)

print("\n" + "="*50)
print("✅ Arize Phoenix успешно запущен и трейсинг жестко привязан!")
print(f"🔗 Кликните по ссылке для просмотра трейсов: {session.url}")
print("="*50 + "\n")

Запуск Arize Phoenix...


/usr/lib/python3.12/contextlib.py:144: SAWarning: Skipped unsupported reflection of expression-based index ix_cumulative_llm_token_count_total
  next(self.gen)
/usr/lib/python3.12/contextlib.py:144: SAWarning: Skipped unsupported reflection of expression-based index ix_latency
  next(self.gen)


🌍 To view the Phoenix app in your browser, visit https://dqkp30lqndm2-496ff2e9c6d22116-6006-colab.googleusercontent.com/
📖 For more information on how to use Phoenix, check out https://arize.com/docs/phoenix
Применяем патч nest_asyncio для работы LangGraph...
🔭 OpenTelemetry Tracing Details 🔭
|  Phoenix Project: langgraph-contract-reviewer
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: http://127.0.0.1:6006/v1/traces
|  Transport: HTTP + protobuf
|  Transport Headers: {}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  ⚠️ WARNING: It is strongly advised to use a BatchSpanProcessor in production environments.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.


✅ Arize Phoenix успешно запущен и трейсинг жестко привязан!
🔗 Кликните по ссылке для просмотра трейсов: https://dqkp30lqndm2-496ff2e9c6d22116-6006-colab

In [3]:
from google.colab import output

print("==================================================")
print("🚀 АЛЬТЕРНАТИВНЫЙ ДОСТУП К PHOENIX")
print("Открываем интерфейс через встроенный прокси Colab...")
print("==================================================")

# 6006 — это стандартный порт, на котором крутится Arize Phoenix
output.serve_kernel_port_as_window(6006, path="/")

🚀 АЛЬТЕРНАТИВНЫЙ ДОСТУП К PHOENIX
Открываем интерфейс через встроенный прокси Colab...
Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

*Внимание: Оставьте вкладку с Phoenix открытой, туда в реальном времени посыплются логи работы нашего графа.*

##Тест LangGraph и проверка трейсов
Давайте запустим наш граф и убедимся, что трейсы появились.

In [4]:
from typing import TypedDict, Annotated
import operator
from langgraph.graph import StateGraph, START, END
from langchain_community.chat_models import ChatOllama
from langchain_core.messages import HumanMessage

# Инициализируем модель
llm = ChatOllama(model="qwen2.5-coder:1.5b", temperature=0.1)

class ContractState(TypedDict):
    contract_text: str
    issues: Annotated[list, operator.add]
    review_count: int

def extract_issues(state: ContractState):
    print("  [LangGraph] 🕵️ Агент ищет риски...")
    prompt = f"Найди 1 юридический риск в тексте: {state['contract_text']}. Верни только риск."
    response = llm.invoke([HumanMessage(content=prompt)])
    return {"issues":[response.content], "review_count": state.get("review_count", 0) + 1}

builder = StateGraph(ContractState)
builder.add_node("extract_issues", extract_issues)
builder.add_edge(START, "extract_issues")
builder.add_edge("extract_issues", END)

langgraph_microservice = builder.compile()

print("Запускаем граф...")
langgraph_microservice.invoke({"contract_text": "Сторона А может забрать квартиру за долг в 1 рубль.", "review_count": 0})
print("✅ Граф отработал. Зайдите в Phoenix по ссылке из предыдущей ячейки — трейс должен появиться!")

/tmp/ipykernel_1300/1862728845.py:8: LangChainDeprecationWarning: The class `ChatOllama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import ChatOllama``.
  llm = ChatOllama(model="qwen2.5-coder:1.5b", temperature=0.1)


Запускаем граф...
  [LangGraph] 🕵️ Агент ищет риски...
✅ Граф отработал. Зайдите в Phoenix по ссылке из предыдущей ячейки — трейс должен появиться!


##Подключение к РЕАЛЬНОМУ Dify Cloud
Инструкция:
Зайдите на https://cloud.dify.ai/.

1.   Создайте пустое приложение типа "Chatbot" (Чат-бот).
2.   В меню слева найдите раздел "API Access" (Доступ по API).
3.   Нажмите "API Key" в правом верхнем углу и сгенерируйте ключ.
4.   Вставьте его в переменную DIFY_API_KEY ниже.

In [5]:
import requests
import json

# --- ВАШ КЛЮЧ ИЗ ОБЛАКА DIFY ---
DIFY_API_KEY = "app-wBel1Vt9k7TJ0Weipu044had"

DIFY_API_URL = "https://api.dify.ai/v1/chat-messages"

def ask_real_dify_cloud(user_query: str) -> str:
    """Делает реальный HTTP POST запрос в облако Dify."""

    headers = {
        "Authorization": f"Bearer {DIFY_API_KEY}",
        "Content-Type": "application/json"
    }

    payload = {
        "inputs": {},              # Переменные, если вы настроили их в Dify
        "query": user_query,       # Вопрос пользователя
        "response_mode": "blocking", # Ждать полного ответа (не streaming)
        "user": "colab-user-123"   # Уникальный ID пользователя для аналитики Dify
    }

    print(f"[Dify Cloud] Отправка запроса: '{user_query}'...")
    response = requests.post(DIFY_API_URL, headers=headers, json=payload)

    if response.status_code == 200:
        result = response.json()
        return result.get("answer", "Нет ответа")
    else:
        return f"Ошибка API Dify: {response.status_code} - {response.text}"

# Тестируем реальное облако!
# ВНИМАНИЕ: Если вы не вставили валидный ключ, будет ошибка 401 Unauthorized
print("\nОтвет от реального Dify Cloud:")
print(ask_real_dify_cloud("Привет, Dify! Какая у тебя базовая модель?"))


Ответ от реального Dify Cloud:
[Dify Cloud] Отправка запроса: 'Привет, Dify! Какая у тебя базовая модель?'...
Привет! Я основан на модели GPT-3.5 от OpenAI. Как я могу помочь тебе сегодня?


##Открываем интерфейс Open WebUI прямо из Colab!
Мы уже запустили сервер Open WebUI в Ячейке 1 на порту 8080. Google Colab позволяет создать прокси-ссылку для доступа к локальным портам. Выполните эту ячейку, и вы получите рабочий UI!

In [9]:
from google.colab import output

print("==================================================")
print("🌐 ДОСТУП К OPEN WEBUI")
print("Кликните по ссылке ниже, чтобы открыть полноценный интерфейс.")
print("При первом входе он попросит создать аккаунт (данные сохраняются только в рамках этой сессии Colab).")
print("==================================================")

# Пробрасываем порт 8080 в виде кликабельной ссылки-фрейма
# output.serve_kernel_port_as_window(8080, path="/")

# Добавляем путь, который часто использует WebUI
output.serve_kernel_port_as_window(8080, path="/webui")
# ИЛИ попробуйте так:
# output.serve_kernel_port_as_window(8080, path="/auth")

🌐 ДОСТУП К OPEN WEBUI
Кликните по ссылке ниже, чтобы открыть полноценный интерфейс.
При первом входе он попросит создать аккаунт (данные сохраняются только в рамках этой сессии Colab).
Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

ВОЗМОЖНО НЕ ЗАВЕДЁТСЯ!

Ошибка **404 Not Found** при использовании `output.serve_kernel_port_as_window(8080)` означает, что прокси-сервер Colab успешно "пробросил" запрос до вашего сервера (раз он не выдал 502 или 504), но Open WebUI не нашел контент по корневому пути `/`.

### Почему это происходит?
1. **Пути в Open WebUI:** Open WebUI часто требует обращения не к корню `/`, а к конкретным эндпоинтам.
2. **Время запуска:** Open WebUI — тяжелое приложение на FastAPI/SvelteKit. Если вы вызываете `serve_kernel_port_as_window` сразу после старта процесса, он еще не успел инициализировать маршрутизацию.
3. **Конфликт параметров:** Возможно, при запуске `open-webui serve` нужно явно указать `WEBUI_URL` или `/` path.

### Решение: принудительный запуск через `pyngrok` (Самый надежный способ)

Метод `serve_kernel_port_as_window` в Colab капризный. Давайте используем **`ngrok`**, который создает "честный" туннель до вашего сервера. Это работает в 100% случаев, так как это прямое соединение, а не проксирование через Google.

#### 1. Переустановите и запустите через `ngrok` (вместо старой команды запуска Open WebUI):
Вставьте это в ячейку, где у вас запускается Open WebUI:

```python
!pip install -q pyngrok
from pyngrok import ngrok

# ВАЖНО: вам нужен бесплатный токен с сайта ngrok.com (напишите его ниже)
NGROK_TOKEN = "ВАШ_ТОКЕН_NGROK"
ngrok.set_auth_token(NGROK_TOKEN)

# Запускаем туннель
public_url = ngrok.connect(8080)
print(f"🔗 Ссылка для доступа к Open WebUI: {public_url.public_url}")

# Теперь запускаем Open WebUI
env = os.environ.copy()
env["OLLAMA_BASE_URL"] = "http://127.0.0.1:11434"
# Добавляем --webui-url, чтобы он знал, что он за прокси
subprocess.Popen(["open-webui", "serve", "--port", "8080", "--host", "0.0.0.0"],
    env=env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
```

### Почему 404?
Потому что SvelteKit (фронтенд WebUI) требует, чтобы все статические файлы (JS/CSS) грузились корректно. Если Google Colab проксирует только `/`, а `index.html` лежит внутри — вы получите ошибку.

**Рекомендация для занятия:**
Если `ngrok` (с токеном) не заработает (из-за сетевых ограничений Colab), значит **Open WebUI внутри контейнера в Colab — это слишком тяжело**.

**Альтернатива:**
Не тратьте время на отладку фронтенда, который "не лезет" в Colab.
1. **Фокус на Dify:** В Dify вы можете создать "Chatbot App" и опубликовать его как `Public URL` (в настройках Dify есть кнопка "Publish"). Это даст вам готовую ссылку на веб-чат, которую **не нужно пробрасывать из Colab**.
2. **Phoenix:** Оставьте как основной инструмент наблюдения, он уже работает.
3. **WebUI:** Используйте его как локальную установку на ноутбуках, а не внутри Colab. Это стандартная практика для Enterprise-обучения: "Сервер крутится в облаке (Colab), фронтенды открываем у себя".

ДОМАШНЕЕ ЗАДАНИЕ

### 🎓 Инструкция: "Подключаем фронтенд к облачному ИИ"

Мы перенесли "мозги" (LLM и API-шлюз) в Google Colab, а интерфейс запускаем прямо у вас на компьютере. Это экономит оперативную память Colab и дает вам полнофункциональный Open WebUI.

#### Шаг 1: Подготовка вашего ноутбука
Вам понадобится Docker (убедитесь, что Docker Desktop запущен).
1. Откройте терминал (PowerShell или Terminal в macOS/Linux).
2. Выполните команду для запуска Open WebUI в Docker:

```bash
docker run -d -p 3000:8080 --add-host=host.docker.internal:host-gateway -v open-webui:/app/backend/data --name open-webui ghcr.io/open-webui/open-webui:main
```
*   `-p 3000:8080` — веб-интерфейс будет доступен по адресу `http://localhost:3000`.
*   `--add-host` — позволяет контейнеру "видеть" ваш хост (нужно для корректной работы сервисов).

#### Шаг 2: Получение доступа к Colab (Туннель)
Ваш Colab сейчас запущен в облаке Google, а ваш ноутбук — дома. Чтобы их связать, нам нужно "прокинуть" туннель из Colab прямо к вам.

В Colab, **в ячейке с Ollama/Dify**, добавьте этот код (используем `ngrok` для проброса порта):

```python
!pip install -q pyngrok
from pyngrok import ngrok

# ВАЖНО: получите бесплатный токен на ngrok.com
NGROK_TOKEN = "ВАШ_ТОКЕН_NGROK"
ngrok.set_auth_token(NGROK_TOKEN)

# Пробрасываем порт Ollama (11434) или Dify (если используете его API)
http_tunnel = ngrok.connect(11434)
print(f"🔗 СКОПИРУЙТЕ ЭТУ ССЫЛКУ: {http_tunnel.public_url}")
```

#### Шаг 3: Соединяем интерфейс с облаком
Теперь идем в интерфейс Open WebUI на вашем ноутбуке:

1. Откройте `http://localhost:3000`.
2. Зарегистрируйтесь (создается локальный аккаунт, данные никуда не уходят).
3. Нажмите на **иконку профиля** (внизу слева) -> **Settings** (Настройки).
4. Перейдите в **Connections** (или **Ollama** в старых версиях).
5. В поле **Ollama Base URL** вставьте ссылку, которую выдал `ngrok` из ячейки Colab (например: `https://abcd-1234.ngrok-free.app`).
   *   *Важно:* добавьте `https://` и убедитесь, что ссылка заканчивается на `.app`.
6. Нажмите кнопку "Refresh" или "Update".

#### Шаг 4: Проверка связи
1. Вернитесь на главный экран чата (иконка "плюс" или "чат").
2. В выпадающем списке моделей (вверху страницы) вы должны увидеть `qwen2.5-coder:1.5b`.
3. Напишите: "Привет, ты работаешь через облако?"
4. Если ответ пришел — **вы успешно развернули распределенную Sovereign AI архитектуру!**